# Анализ данных домена Marketplace за последние 30 дней

В домене Marketplace собраны логи пользователей на маркетплейсе: просмотры, клики, лайки и переходы к покупке. События снабжены контекстами о том, где они происходили: в поиске, каталоге, на главной странице, в рекомендациях (u2i, i2i). Для товаров доступен каталог с брендом, embedding и категориями, но не для всех товаров категории заполнены. https://habr.com/ru/companies/tbank/articles/950696/

In [1]:
import os
import pandas as pd
import numpy as np
import glob
from scipy.stats import zscore

In [2]:
events_dir = "../data/raw/T-bank dataset full/T-ECD-small/dataset/small/marketplace/events"

In [3]:
# Получаем список всех файлов .pq
event_files = [f for f in os.listdir(events_dir) if f.endswith('.pq')]

In [4]:
# Отсортируем по номеру дня
event_files_sorted = sorted(event_files, reverse=True)

In [5]:
# Возьмем последние 30 файлов
last_30_files = event_files_sorted[:30]

In [6]:
# Считываем и объединяем данные
dfs = []
for fname in last_30_files:
    filepath = os.path.join(events_dir, fname)
    df = pd.read_parquet(filepath)
    dfs.append(df)

In [7]:
# Объединенный DataFrame за последние 30 дней
recent_events = pd.concat(dfs, ignore_index=True)

In [8]:
users_df = pd.read_parquet('../data/raw/T-bank dataset full/T-ECD-small/dataset/small/users.pq')

In [9]:
items_df = pd.read_parquet('../data/raw/T-bank dataset full/T-ECD-small/dataset/small/marketplace/items.pq')

In [10]:
brands_df = pd.read_parquet('../data/raw/T-bank dataset full/T-ECD-small/dataset/small/brands.pq', engine='fastparquet')

# Исследуем таблицу recent_events

In [11]:
recent_events.head()

,timestamp,user_id,item_id,subdomain,action_type,os
0,113011200183108,73774100,nfmcg_18200186,other,view,android
1,113011200333142,38161973,nfmcg_14129567,other,view,android
2,113011200570986,52431976,nfmcg_4396804,search,view,android
3,113011200656523,75059116,nfmcg_27122678,i2i,view,other
4,113011200923703,10628666,nfmcg_6136132,other,view,android


In [12]:
recent_events.shape

(17790404, 6)

In [13]:
print(f'Уникальных пользователей: {recent_events.user_id.nunique()}')
print(f'Уникальных айтемов: {recent_events.item_id.nunique()}')

Уникальных пользователей: 660068
Уникальных айтемов: 844803


In [14]:
recent_events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17790404 entries, 0 to 17790403
Data columns (total 6 columns):
 #   Column       Dtype 
---  ------       ----- 
 0   timestamp    int64 
 1   user_id      uint64
 2   item_id      object
 3   subdomain    object
 4   action_type  object
 5   os           object
dtypes: int64(1), object(4), uint64(1)
memory usage: 814.4+ MB


#### Распределение операционных систем

In [15]:
recent_events['os'].unique()

array(['android', 'other', 'ios'], dtype=object)

Вычислим сколько уникальных юзеров взаимодействовало с операционной системой

In [16]:
recent_events.groupby('os', dropna=False)['user_id'].nunique().sort_values(ascending=False)

os
android    633212
ios        254380
other      205827
Name: user_id, dtype: int64

Из таблицы видно, что чаще всего используемая платформа - android

In [17]:
recent_events['action_type'].unique()

array(['view', 'click', 'clickout', 'like'], dtype=object)

#### Распределение типов действий

Вычислим распределение количества действий в выбранном периоде

In [18]:
recent_events.groupby('action_type', dropna=False)['user_id'].count().sort_values(ascending=False)

action_type
view        17279844
click         431008
clickout       73489
like            6063
Name: user_id, dtype: int64

Самое частое действие - просмотр. Из значимых действий самое частое - click.

Вычислим распределение количества уникальных пользователей по типу событий

In [19]:
recent_events.groupby('action_type', dropna=False)['user_id'].nunique().sort_values(ascending=False)

action_type
view        658422
click       105023
clickout     25136
like          2450
Name: user_id, dtype: int64

Самое частое действие - просмотр. Из значимых действий самое частое - click.

In [20]:
# Пользователи, у которых есть click, clickout или like, но нет view в нашем периоде
users_with_actions = recent_events[recent_events['action_type'].isin(['click', 'clickout', 'like'])]['user_id'].unique()
users_with_views = recent_events[recent_events['action_type'] == 'view']['user_id'].unique()

users_without_views = set(users_with_actions) - set(users_with_views)
print(f"Пользователи с действиями, но без views в анализируемом периоде: {len(users_without_views)}")

Пользователи с действиями, но без views в анализируемом периоде: 1646


#### Распределение событий по контекстам, где они происходили (рекомендации, каталог, поиск, похожие товары)

In [21]:
recent_events['subdomain'].unique()

array(['other', 'search', 'i2i', 'u2i', 'catalog', None], dtype=object)

Распределение количества уникальных уникальных пользователей по контекстам

In [22]:
recent_events.groupby('subdomain', dropna=False)['user_id'].nunique().sort_values(ascending=False)

subdomain
u2i        493783
other      328767
i2i         78744
catalog     53774
search      29042
NaN          2450
Name: user_id, dtype: int64

Из таблицы видно, что наибольшее количество взаимодействий пользователей с товаром произошло когда товар был показан в блоке персональных рекомендаций, сформированных специально для этого пользователя.

### Работа со временем. Предположение - время задано в микросекундах со сдвигом

In [23]:
recent_events['timestamp'].min(), recent_events['timestamp'].max()

(np.int64(110505600037405), np.int64(113097598175560))

In [24]:
# конвертируем в datetime (получим примерно 1973 год)
recent_events['datetime_wrong_year'] = pd.to_datetime(recent_events['timestamp'], unit='us')

In [25]:
# желаемая и фактическая даты начала
# .normalize() убирает время, оставляя только дату
desired_start_date = pd.to_datetime('2025-01-01')
actual_start_date = recent_events['datetime_wrong_year'].min().normalize() 

In [26]:
# сдвиг
time_offset = desired_start_date - actual_start_date

In [27]:
# применяем сдвиг ко всем датам
recent_events['datetime_corrected'] = recent_events['datetime_wrong_year'] + time_offset

In [28]:
print(recent_events[['timestamp', 'datetime_wrong_year', 'datetime_corrected']].sort_values(by='datetime_corrected').head())

                timestamp        datetime_wrong_year  \
16989295  110505600037405 1973-07-03 00:00:00.037405   
16989296  110505600037405 1973-07-03 00:00:00.037405   
16989297  110505600219769 1973-07-03 00:00:00.219769   
16989298  110505600655919 1973-07-03 00:00:00.655919   
16989299  110505601306735 1973-07-03 00:00:01.306735   

                 datetime_corrected  
16989295 2025-01-01 00:00:00.037405  
16989296 2025-01-01 00:00:00.037405  
16989297 2025-01-01 00:00:00.219769  
16989298 2025-01-01 00:00:00.655919  
16989299 2025-01-01 00:00:01.306735  


In [29]:
print(recent_events[['timestamp', 'datetime_wrong_year', 'datetime_corrected']].sort_values(by='datetime_corrected').tail())

              timestamp        datetime_wrong_year         datetime_corrected
533349  113097596634823 1973-08-01 23:59:56.634823 2025-01-30 23:59:56.634823
533350  113097596970248 1973-08-01 23:59:56.970248 2025-01-30 23:59:56.970248
533351  113097597225981 1973-08-01 23:59:57.225981 2025-01-30 23:59:57.225981
533352  113097598066187 1973-08-01 23:59:58.066187 2025-01-30 23:59:58.066187
533353  113097598175560 1973-08-01 23:59:58.175560 2025-01-30 23:59:58.175560


In [30]:
print(recent_events['datetime_corrected'].max() - recent_events['datetime_corrected'].min())

29 days 23:59:58.138155


Вывод:  скорее всего, время действительно задано в микросекундах со сдвигом.

# Исследуем данные из таблицы users_df

In [31]:
users_df.head()

,user_id,socdem_cluster,region
0,77309558,21.0,2.0
1,72517894,10.0,90.0
2,86699708,9.0,9.0
3,54241043,17.0,58.0
4,23591057,17.0,4.0


In [32]:
users_df.shape

(3500000, 3)

In [33]:
users_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3500000 entries, 0 to 3499999
Data columns (total 3 columns):
 #   Column          Dtype  
---  ------          -----  
 0   user_id         uint64 
 1   socdem_cluster  float64
 2   region          float64
dtypes: float64(2), uint64(1)
memory usage: 80.1 MB


In [34]:
users_df['socdem_cluster'].nunique()

21

In [35]:
users_df['region'].nunique()

90

#### Распределение по социо-демографическим кластерам (первые пять наиболее заполненнных)

In [36]:
users_df.groupby('socdem_cluster', dropna=False)['user_id'].nunique().sort_values(ascending=False).head()

socdem_cluster
17.0    473629
20.0    420944
12.0    416287
9.0     340144
21.0    331140
Name: user_id, dtype: int64

#### Распределение по регионам

In [37]:
users_df.groupby('region', dropna=False)['user_id'].nunique().sort_values(ascending=False).head()

region
2.0     551358
60.0    249920
90.0    181623
18.0    137250
24.0    114992
Name: user_id, dtype: int64

# Исследуем данные из таблицы items_df

In [38]:
items_df.head()

,item_id,brand_id,category,subcategory,price,embedding
0,nfmcg_10000001,137356,None,None,6.063640,"[-0.07085289, 0.032460444, 0.009126052, 0.0377..."
1,nfmcg_10000012,53389,None,None,0.960436,"[-0.09109873, 0.043168057, -0.027675372, 0.031..."
2,nfmcg_1000002,34153,"Fashion Accessories, Tech Add-ons, and Style E...","Hats, Scarves, and Shawls",-1.793113,"[-0.05653296, 0.08287799, -0.06381639, 0.09553..."
3,nfmcg_10000034,39543,None,None,3.416755,"[-0.11258836, 0.04333279, 0.029613094, -0.0040..."
4,nfmcg_10000039,82320,"Fashion Accessories, Tech Add-ons, and Style E...",Jewelry and Costume Jewelry,4.293433,"[-0.15720145, 0.02623378, 0.005749651, 0.06972..."


In [39]:
items_df.shape

(2325409, 6)

In [40]:
items_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2325409 entries, 0 to 2325408
Data columns (total 6 columns):
 #   Column       Dtype  
---  ------       -----  
 0   item_id      object 
 1   brand_id     uint64 
 2   category     object 
 3   subcategory  object 
 4   price        float64
 5   embedding    object 
dtypes: float64(1), object(4), uint64(1)
memory usage: 106.4+ MB


In [41]:
items_df['category'].unique()

array([None, 'Fashion Accessories, Tech Add-ons, and Style Enhancements',
       'Miscellaneous Goods (Uncategorized)',
       'Cosmetics, Personal Care, and Health Maintenance Products',
       'Footwear of All Types',
       'Home/Office Furniture and Interior Decor',
       'Sports Equipment, Gear, and Outdoor Activity Accessories',
       'Electronic Devices and Gadgets',
       'Construction and Renovation Materials, Tools, and Equipment',
       'Home Improvement and Countryside Retreat Essentials',
       "Children's Products and Childcare Items",
       'Outerwear, Casual Apparel, and Specialized Workwear',
       'Household Electrical Appliances', 'Foodstuffs and Beverages',
       'Office Supplies and Educational Materials',
       'Pharmaceuticals and Medical Supplies',
       'Hobby, Creative, and Leisure Products',
       'Cleaning Supplies and Everyday Household Items',
       'Pet Supplies: Food, Accessories, and Grooming Products',
       'Automotive Accessories, Spare 

In [42]:
items_df.groupby('category', dropna=False)['item_id'].nunique().sort_values(ascending=False)

category
NaN                                                            966395
Miscellaneous Goods (Uncategorized)                            266146
Cosmetics, Personal Care, and Health Maintenance Products      188412
Fashion Accessories, Tech Add-ons, and Style Enhancements      167811
Outerwear, Casual Apparel, and Specialized Workwear            153223
Footwear of All Types                                          103680
Electronic Devices and Gadgets                                  84282
Home Improvement and Countryside Retreat Essentials             83707
Construction and Renovation Materials, Tools, and Equipment     83584
Home/Office Furniture and Interior Decor                        64919
Household Electrical Appliances                                 61762
Sports Equipment, Gear, and Outdoor Activity Accessories        36214
Hobby, Creative, and Leisure Products                           13292
Children's Products and Childcare Items                         13161
Automotive 

In [43]:
items_df['subcategory'].nunique()

192

192 уникальные подкатегории

In [44]:
items_df.groupby('subcategory', dropna=False)['item_id'].nunique().sort_values(ascending=False)

subcategory
NaN                                        1233023
Jewelry and Costume Jewelry                  99751
Women's Clothing                             82778
Women's Footwear                             62212
Men's Clothing                               46759
                                            ...   
Veterinary Medications and Products              7
Educational and Instructional Materials          7
Meat and Sausage Products                        6
Farm Produce                                     6
Horse and Equestrian Supplies                    1
Name: item_id, Length: 193, dtype: int64

# Посмотрим на данные из таблицы brands_df

In [45]:
brands_df.head()

,brand_id,embedding
0,4,None
1,34,None
2,45,None
3,46,None
4,51,None


In [46]:
brands_df.shape

(24513, 2)

In [47]:
brands_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24513 entries, 0 to 24512
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   brand_id   24513 non-null  uint64
 1   embedding  6559 non-null   object
dtypes: object(1), uint64(1)
memory usage: 383.1+ KB


# Анализ пропусков

## Пропущенные значения в датасете users_df (совпадает во всех доменах)

In [48]:
users_df.isna().sum()

user_id               0
socdem_cluster     5153
region            58917
dtype: int64

In [49]:
def missing_value_rate(df, column):
  print(f'Доля пропущенных значений в колонке {column}:  {(df[column].isna().sum() / users_df.shape[0]):.4f}')

In [50]:
for column in users_df.columns:
  missing_value_rate(users_df, column)

Доля пропущенных значений в колонке user_id:  0.0000
Доля пропущенных значений в колонке socdem_cluster:  0.0015
Доля пропущенных значений в колонке region:  0.0168


Возможные причины возникновения пропусков:

* Необязательные поля при регистрации

* Система не смогла определить местоположение пользователя, возможно пользователь не дал разрешение на определение местоположения

* Технические ошибки при сборе данных

In [51]:
print(users_df[(users_df['socdem_cluster'].isna()) & (users_df['region'].isna())].shape[0])
print(users_df[(users_df['socdem_cluster'].isna()) & (users_df['region'].isna())].shape[0] / users_df.shape[0])

4667
0.0013334285714285713


Итого у нас есть 4667 записей пользователей, где пропуски в обеих колонках socdem_cluster и region. Это около 1,3%.

In [52]:
# проверим дубликаты
users_df.duplicated().sum()

np.int64(0)

In [53]:
print(f'Минимальное значение в колонках socdem_cluster и region соответвенно: {users_df['socdem_cluster'].min()} и {users_df['region'].min()}')

Минимальное значение в колонках socdem_cluster и region соответвенно: 0.0 и 0.0


### Возможная стратегия работы с пропусками

Процент пропусков очень мал. 

* Технически можно удалить строки с пропущенными значениями в таблице users_df. Но тогда мы должны будем удалить этих пользователей из таблицы events_df, которую мы будем использовать для посторения матрицы взаимодействий и уже на этой основе строить прогноз. 

* Колонки socdem_cluster и region это категориальные данные, закодированные числами. Можно заполнить пропущенные значения специальным зарезервированным числом, которое не встречается среди реальных кодов регионов и кластеров. Таким образом, мы сохраним всю историю взаимодействий c этими айтемами в events_df. Можно использовать число -1, так как минимальное значение в указанных колонках равно ноль.

## Пропущенные значения в items_df

In [54]:
items_df.isna().sum()

item_id              0
brand_id             0
category        966395
subcategory    1233023
price             2882
embedding           73
dtype: int64

In [55]:
for column in items_df.columns:
  missing_value_rate(items_df, column)

Доля пропущенных значений в колонке item_id:  0.0000
Доля пропущенных значений в колонке brand_id:  0.0000
Доля пропущенных значений в колонке category:  0.2761
Доля пропущенных значений в колонке subcategory:  0.3523
Доля пропущенных значений в колонке price:  0.0008
Доля пропущенных значений в колонке embedding:  0.0000


### Возможные причины возникновения пропусков

* Для `category` и `subcategory` - товар могли добавить в базу данных, но еще не успели присвоить ему категорию.

* Некоторые товары могут не подходить ни под одну из существующих категорий, и поле остается пустым.

* Поставшики прислали неполную информацию.

* Для `price` - это могут быть товары не для продажи, а, например, промо-материалы.

* Некоторые товары могут не иметь публичной цены, а она узнается по запросу.

* Часть комплекта: товар может продаваться только в составе набора, и у него нет индивидуальной цены.

### Возможная стратегия работы с пропусками

* в колонках `category` и `subcategory` (категориальные данные) заменить на 'unknown'.Таким образом, мы сохраняем всю его ценную историю взаимодействий в `events_df`.
  
* Это лучше, чем заполнять самой частой категорией, так как это может внести смещение в данные.

* Колонка `price`. Можно заполнить нулем, если мы можем предположить, что товар, например, акционный, бесплатный. Но это частный случай и у нас нет данных для выдвижения такого предположения. Лучше заполнить медианой. Медиана более устойчива к выбросам, поэтому она лучше отражает "типичную" цену.

## Пропущенные значения в events_df

In [56]:
recent_events.isna().sum()

timestamp                 0
user_id                   0
item_id                   0
subdomain              6063
action_type               0
os                        0
datetime_wrong_year       0
datetime_corrected        0
dtype: int64

In [57]:
recent_events['subdomain'].unique()

array(['other', 'search', 'i2i', 'u2i', 'catalog', None], dtype=object)

subdomain - область, на которой происходило взаимодействие (рекомендации, каталог, поиск, оформление заказа, кампания).

In [58]:
missing_value_rate(recent_events, 'subdomain')

Доля пропущенных значений в колонке subdomain:  0.0017


### Возможная стратегия работы с пропусками

* в колонке `subdomain` заменить на 'unknown'.

## Пропущенные значения в brands_df (совпадает во всех доменах)

In [59]:
brands_df.isna().sum()

brand_id         0
embedding    17954
dtype: int64

In [60]:
print(f'Доля пропущенных значений в колонке embedding:  {(brands_df['embedding'].isna().sum() / brands_df.shape[0]):.4f}')

Доля пропущенных значений в колонке embedding:  0.7324


Более 73% брендов не имеют эмбеддинга. Возможные причины:

* Пропущенные эмбеддинги могут говорить о холодных брендах, а могут и не говорить.

* Эмбеддинги могут пересчитывать не в реальном времени, а, например, раз в неделю. Пропуски могут соответствовать брендам, которые были добавлены после последнего обновления и еще ждут своей очереди на обработку.

### Возможная стратегия работы с пропусками

* Можно удалить колонку

* Можно оставить и попробовать что-то сделать в рамках нейросетевой модели

* Можно посмотреть временную связь на всех данных


**Заметка про работу с холодным стартом на будущее** (из лекция https://vkvideo.ru/video-123851409_456239391?list=ln-rYpN16j7stPziuqnla)

* Рекомендовать популярные

* Построить модель на соцдеме или других данных (Например, женщинам рекомендовать женскую одежду, мужчинам мужскую и тд). То есть построить набор популярных товаров, характерных для этого соцдема.

* Провести онбординг (некоторые сервисы при регистрации спрашивают, укажите что вам нравится)

# Анализ до заполнения пропусков

### Анализ users_id (совпадает во всех доменах)

In [61]:
print(f'Уникальных пользователей: {users_df['user_id'].nunique()}')
print(f'Уникальных регионов: {users_df['region'].nunique()}')
print(f'Уникальных социо-демографических кластеров: {users_df['socdem_cluster'].nunique()}')

Уникальных пользователей: 3500000
Уникальных регионов: 90
Уникальных социо-демографических кластеров: 21


In [62]:
# Посмотрим, сколько пользователей в каждом регионе
users_df['region'].value_counts(dropna=False)

region
2.0     551358
60.0    249920
90.0    181623
18.0    137250
24.0    114992
         ...  
65.0      1687
30.0      1125
69.0       915
39.0       836
20.0       338
Name: count, Length: 91, dtype: int64

In [63]:
# Посмотрим, сколько пользователей в каждом кластере
users_df['socdem_cluster'].value_counts(dropna=False)

socdem_cluster
17.0    473629
20.0    420944
12.0    416287
9.0     340144
21.0    331140
10.0    266045
19.0    261186
4.0     254800
0.0     205299
5.0     148037
7.0     140750
18.0    105353
13.0     40572
6.0      36130
1.0      18629
2.0      12134
16.0      9859
8.0       5825
NaN       5153
3.0       3900
11.0      2637
15.0      1547
Name: count, dtype: int64

In [64]:
# Преобразуем столбцы в категориальный тип
users_df_with_cat = users_df.copy()
users_df_with_cat['socdem_cluster'] = users_df['socdem_cluster'].astype('category')
users_df_with_cat['region'] = users_df['region'].astype('category')

In [65]:
# И применим describe
print(users_df_with_cat[['socdem_cluster', 'region']].describe())

        socdem_cluster     region
count        3494847.0  3441083.0
unique            21.0       90.0
top               17.0        2.0
freq          473629.0   551358.0


### Анализ items_df

In [66]:
dist_without_na = items_df['price'].describe()
print(dist_without_na.apply(lambda x: f'{x:.5f}'))

count    2322527.00000
mean           0.48980
std            2.45504
min          -10.00000
25%           -1.21974
50%            0.42576
75%            2.19402
max           10.00000
Name: price, dtype: object


* Cреднее значение и медиана близки друг к другу, значит распределение относительно симметричное.

* Имеет место правостороннее распределение с положительной асимметрией, где среднее значение 0.49 и медиана 0.43 смещены в положительную область, что указывает на преобладание цен выше условного нуля. Четверть значений остаются отрицательными (25-й перцентиль: -1.22), при этом основная масса данных сосредоточена в интервале от -1.22 до 2.19. 

Применим метод z-score для поиска выбросов

In [67]:
price_without_na = items_df['price'].dropna()

In [68]:
z_scores = zscore(price_without_na) 

In [69]:
z_scores

array([ 2.27036813,  0.19170071, -0.9298913 , ...,  0.11589517,
       -0.55253731,  0.06045728], shape=(2322527,))

In [70]:
# Находим индексы выбросов в данных без пропусков.
outlier_indices = np.where(np.abs(z_scores) > 3)[0]

In [71]:
print(f"Найдено выбросов: {len(outlier_indices)}")

Найдено выбросов: 5736


In [72]:
# Смотрим на сами значения выбросов
outliers = price_without_na.iloc[outlier_indices]
print("\nПримеры найденных выбросов:")
print(outliers)


Примеры найденных выбросов:
247        7.946068
332        8.471466
519        7.861344
584        8.575839
1362       8.381851
             ...   
2320875   -8.099623
2321166   -7.901794
2321535   -7.675526
2322584   -7.915947
2323904    7.881762
Name: price, Length: 5736, dtype: float64


# Покажем варианты заполнения пропусков

## Обработка users_df (совпадает во всех доменах)
В таблице users_df заполняем пропуски в закодированных категориальных признаках region и socdem_cluster числом -1. Мы это можем сделать, так как минимальные значения этих признаков равны нулю.

In [73]:
users_df['region'] = users_df['region'].fillna(-1)
users_df['socdem_cluster'] = users_df['socdem_cluster'].fillna(-1)

In [74]:
# Меняем тип данных на целочисленный
users_df['region'] = users_df['region'].astype(int)
users_df['socdem_cluster'] = users_df['socdem_cluster'].astype(int)

In [75]:
print('Пропуски в users_df после обработки:')
print(users_df.isnull().sum())

Пропуски в users_df после обработки:
user_id           0
socdem_cluster    0
region            0
dtype: int64


## Обработка items_df

In [76]:
# Заполняем категориальные пропуски строкой 'unknown'
items_df['category'] = items_df['category'].fillna('unknown')
items_df['subcategory'] = items_df['subcategory'].fillna('unknown')

In [77]:
# Рассчитываем медианную цену
median_price = items_df['price'].median()
print(f'Медианная цена для заполнения пропусков: {median_price}')

Медианная цена для заполнения пропусков: 0.42575823138244323


In [78]:
items_df['price'].min()

np.float64(-10.0)

В признаке price присутствуют отрицательные значения. Это не выбросы. В описании дата сета price - представляет собой денежную стоимость взаимодействия. Цены в каталогах переведены в промежуток от −10 до 10. 

In [79]:
# Заполняем пропуски в цене медианой
items_df['price'] = items_df['price'].fillna(median_price)

In [80]:
print("Пропуски в items_df после обработки:")
print(items_df.isnull().sum())

Пропуски в items_df после обработки:
item_id         0
brand_id        0
category        0
subcategory     0
price           0
embedding      73
dtype: int64


## Обработка brands_df

Пропуски в embedding пока не трогаем. 

## Анализ цен  из items_df после заполнения пропусков

In [81]:
dist = items_df['price'].describe()
print(dist.apply(lambda x: f'{x:.5f}'))

count    2325409.00000
mean           0.48972
std            2.45352
min          -10.00000
25%           -1.21740
50%            0.42576
75%            2.19184
max           10.00000
Name: price, dtype: object


* Из таблицы видно, что среднее значение и медиана близки друг к другу, значит распределение относительно симметричное.

* Имеет место правостороннее распределение с положительной асимметрией, где среднее значение 0.49 и медиана 0.43 смещены в положительную область, что указывает на преобладание цен выше условного нуля.

* Четверть значений остаются отрицательными (25-й перцентиль: -1.22), при этом основная масса данных сосредоточена в интервале от -1.22 до 2.19.

In [82]:
# Сравним с распределением цен до заполнения пропусков
print(dist_without_na.apply(lambda x: f'{x:.5f}'))

count    2322527.00000
mean           0.48980
std            2.45504
min          -10.00000
25%           -1.21974
50%            0.42576
75%            2.19402
max           10.00000
Name: price, dtype: object


Заполнение пропусков медианой прошло успешно и оказало минимальное, предсказуемое влияние на общую картину распределения цен 

# Анализ events_df

In [83]:
recent_events.head()

,timestamp,user_id,item_id,subdomain,action_type,os,datetime_wrong_year,datetime_corrected
0,113011200183108,73774100,nfmcg_18200186,other,view,android,1973-08-01 00:00:00.183108,2025-01-30 00:00:00.183108
1,113011200333142,38161973,nfmcg_14129567,other,view,android,1973-08-01 00:00:00.333142,2025-01-30 00:00:00.333142
2,113011200570986,52431976,nfmcg_4396804,search,view,android,1973-08-01 00:00:00.570986,2025-01-30 00:00:00.570986
3,113011200656523,75059116,nfmcg_27122678,i2i,view,other,1973-08-01 00:00:00.656523,2025-01-30 00:00:00.656523
4,113011200923703,10628666,nfmcg_6136132,other,view,android,1973-08-01 00:00:00.923703,2025-01-30 00:00:00.923703


#### Определение самых популярных товаров

События типа 'view' фиксируют только факт показа товара пользователю, и многие из них могут быть случайными или малозначимыми (пользователь просто пролистал страницу, товар попал в рекомендацию, но не заинтересовал). Поэтому топ по просмотрам покажет товары, которые чаще всего показывались, но не обязательно были востребованы.

Для выявления действительно популярных и интересных товаров лучше использовать более сильные сигналы, такие как:

'click' — пользователь проявил явный интерес, кликнув на товар.

'like' — проявил позитивную реакцию, добавил в избранное.

'clickout' — намерение к покупке, переход к оформлению или внешнему сайту.

In [84]:
# Вспомним, какие есть типы взаимодействий
recent_events['action_type'].unique()

array(['view', 'click', 'clickout', 'like'], dtype=object)

In [85]:
# Берем только клики и считаем количества кликов для каждого item_id
top_popular_items = recent_events[recent_events['action_type'] == 'click']['item_id'].value_counts()

# Выводим топ-10 самых популярных товаров по кликам
print(top_popular_items.head(10))

item_id
nfmcg_20987893    10323
nfmcg_22178359     7998
nfmcg_13647154     6349
nfmcg_4104717      6310
nfmcg_14380950     6108
nfmcg_8961427      5887
nfmcg_28106008     4453
nfmcg_3698542      4292
nfmcg_22211380     4157
nfmcg_6301488      3679
Name: count, dtype: int64


In [86]:
# Берем только clickout и считаем количества clickout для каждого item_id
top_popular_items = recent_events[recent_events['action_type'] == 'clickout']['item_id'].value_counts()

# Выводим топ-10 самых популярных товаров
print(top_popular_items.head(10))

item_id
nfmcg_20987893    1490
nfmcg_4104717     1090
nfmcg_22178359     705
nfmcg_18200186     698
nfmcg_22437058     674
nfmcg_16225713     646
nfmcg_13647154     635
nfmcg_6136132      628
nfmcg_28106008     627
nfmcg_6613648      553
Name: count, dtype: int64


In [87]:
# Берем только лайки и считаем количества лайков для каждого item_id
top_popular_items = recent_events[recent_events['action_type'] == 'like']['item_id'].value_counts()

# Выводим топ-10 самых популярных товаров по лайкам
print(top_popular_items.head(10))

item_id
nfmcg_20987893    118
nfmcg_4104717      88
nfmcg_18106422     68
nfmcg_13647154     61
nfmcg_28106008     53
nfmcg_22437058     52
nfmcg_6136132      52
nfmcg_9717065      51
nfmcg_8961427      47
nfmcg_22178359     45
Name: count, dtype: int64


Создадим столбец label. Действию view в колонке label поставим в соответсвие 0, а действиям click, clickout, like поставим в соответсвие 1.

In [88]:
recent_events['label'] = recent_events['action_type'].map({'view':0, 'click':1, 'clickout':1, 'like': 1})

In [89]:
recent_events.groupby('action_type')['label'].mean()

action_type
click       1.0
clickout    1.0
like        1.0
view        0.0
Name: label, dtype: float64

In [90]:
# Популярность по количеству уникальных пользователей. 
# Именно такой способ расчета популярности часто используется в рекомендательных системах

# Считаем количество уникальных пользователей, совершивших значимые действия для каждого товара
label_1_recent_events = recent_events[recent_events['label'] == 1]
unique_user_popularity = label_1_recent_events.groupby('item_id')['user_id'].nunique().sort_values(ascending=False)

print(unique_user_popularity.head(10))

item_id
nfmcg_20987893    10604
nfmcg_22178359     7491
nfmcg_4104717      6726
nfmcg_13647154     6638
nfmcg_8961427      5753
nfmcg_14380950     5610
nfmcg_28106008     4849
nfmcg_3698542      4240
nfmcg_22211380     4230
nfmcg_6301488      3869
Name: user_id, dtype: int64


# Взаимосвязи количественной и категориальной переменных

В датасете одна основная количественная переменная - `price`

In [91]:
recent_events.head()

,timestamp,user_id,item_id,subdomain,action_type,os,datetime_wrong_year,datetime_corrected,label
0,113011200183108,73774100,nfmcg_18200186,other,view,android,1973-08-01 00:00:00.183108,2025-01-30 00:00:00.183108,0
1,113011200333142,38161973,nfmcg_14129567,other,view,android,1973-08-01 00:00:00.333142,2025-01-30 00:00:00.333142,0
2,113011200570986,52431976,nfmcg_4396804,search,view,android,1973-08-01 00:00:00.570986,2025-01-30 00:00:00.570986,0
3,113011200656523,75059116,nfmcg_27122678,i2i,view,other,1973-08-01 00:00:00.656523,2025-01-30 00:00:00.656523,0
4,113011200923703,10628666,nfmcg_6136132,other,view,android,1973-08-01 00:00:00.923703,2025-01-30 00:00:00.923703,0


In [92]:
users_df.head()

,user_id,socdem_cluster,region
0,77309558,21,2
1,72517894,10,90
2,86699708,9,9
3,54241043,17,58
4,23591057,17,4


In [93]:
items_df.head()

,item_id,brand_id,category,subcategory,price,embedding
0,nfmcg_10000001,137356,unknown,unknown,6.063640,"[-0.07085289, 0.032460444, 0.009126052, 0.0377..."
1,nfmcg_10000012,53389,unknown,unknown,0.960436,"[-0.09109873, 0.043168057, -0.027675372, 0.031..."
2,nfmcg_1000002,34153,"Fashion Accessories, Tech Add-ons, and Style E...","Hats, Scarves, and Shawls",-1.793113,"[-0.05653296, 0.08287799, -0.06381639, 0.09553..."
3,nfmcg_10000034,39543,unknown,unknown,3.416755,"[-0.11258836, 0.04333279, 0.029613094, -0.0040..."
4,nfmcg_10000039,82320,"Fashion Accessories, Tech Add-ons, and Style E...",Jewelry and Costume Jewelry,4.293433,"[-0.15720145, 0.02623378, 0.005749651, 0.06972..."


In [94]:
# Рассчитываем среднюю, медианную цену и др. для каждого бренда
price_by_brand = items_df.groupby('brand_id')['price'].describe()
print(price_by_brand.sort_values(by='mean', ascending=False))

           count      mean       std       min       25%       50%       75%  \
brand_id                                                                       
21903     2577.0  5.245232  1.680747 -0.125418  4.082462  5.236810  6.412017   
82320     8859.0  5.224603  1.984396 -0.235001  3.599150  5.016587  7.028653   
78943     5753.0  4.858621  1.824220 -2.746199  3.680813  5.145148  6.315990   
70985     7267.0  4.409246  1.806163  1.518843  2.890581  4.188179  5.742984   
38518     1671.0  4.215415  1.399502  0.564942  3.206599  4.183658  5.207582   
...          ...       ...       ...       ...       ...       ...       ...   
87209     2098.0 -2.556432  1.445784 -7.419848 -3.398756 -2.401861 -1.519913   
239720    3285.0 -2.576888  0.930904 -5.534725 -3.201246 -2.631111 -1.985349   
94319       83.0 -2.813621  0.274389 -3.500528 -2.888634 -2.840146 -2.814393   
247765     823.0 -4.580355  1.821099 -6.452926 -6.046263 -5.209547 -3.674645   
166052     398.0 -4.855042  1.012550 -6.

In [95]:
# Рассчитываем среднюю, медианную цену и др. для каждой категории
price_by_brand = items_df.groupby('category')['price'].describe()
print(price_by_brand.sort_values(by='mean', ascending=False).head())

                                                       count      mean  \
category                                                                 
Home/Office Furniture and Interior Decor             64919.0  2.975864   
Professional-Grade Tools and Industrial Equipment      473.0  2.527799   
Household Electrical Appliances                      61762.0  2.162326   
Fashion Accessories, Tech Add-ons, and Style En...  167811.0  1.671725   
Footwear of All Types                               103680.0  1.249630   

                                                         std        min  \
category                                                                  
Home/Office Furniture and Interior Decor            1.936730  -8.178623   
Professional-Grade Tools and Industrial Equipment   3.090084  -4.689742   
Household Electrical Appliances                     2.311974 -10.000000   
Fashion Accessories, Tech Add-ons, and Style En...  2.332411 -10.000000   
Footwear of All Types          

In [96]:
# Объединяем таблицу событий с данными о пользователях
merged_df = pd.merge(recent_events, users_df, on='user_id', how='left')

# присоединяем данные о товарах
merged_df = pd.merge(merged_df, items_df, on='item_id', how='left')

In [97]:
# Рассчитываем описательные статистики цен для каждого соц-дем кластера
price_by_cluster = merged_df.groupby('socdem_cluster', observed=True)['price'].describe()

# Выводим кластеры, предпочитающие товары с высокой средней ценой
print("Статистики цен по социально-демографическим кластерам:")
print(price_by_cluster.sort_values(by='mean', ascending=False))

Статистики цен по социально-демографическим кластерам:
                    count      mean       std       min       25%       50%  \
socdem_cluster                                                                
 12             1862855.0  2.354373  2.604402 -8.511610  0.480142  2.347722   
 21             1021810.0  2.323039  2.700563 -8.511610  0.499214  2.376265   
 17             2900407.0  2.240428  2.516075 -8.511610  0.250577  2.258955   
-1                16437.0  2.199459  2.213447 -5.944676  0.636885  2.260275   
 4              1740665.0  2.101025  2.363551 -8.125625  0.207010  2.222647   
 8                27094.0  2.084260  2.385157 -7.466088  0.557869  2.204123   
 1                37932.0  2.008796  2.509439 -7.475944  0.269912  2.109904   
 13               11530.0  2.004475  2.134527 -6.401113  0.552374  2.095919   
 15                2973.0  2.001670  2.424793 -6.647431  0.238149  2.043378   
 6                16028.0  1.965556  2.100106 -7.473300  0.557616  2.086215 

# Целевая переменная

На основе семинара-лекции: https://vkvideo.ru/video-123851409_456239391?list=ln-rYpN16j7stPziuqnla

У нас есть список релевантных пользователю товаров в обучающей выборке (то есть айдишники товаров, с которым пользователь сделал значимые интеракции на обучающем промежутке времени). И мы хотим предсказать айдишники товаров, с которыми пользователь провзаимодейтсвует на тестовом (отложенном) периоде времени. Для подсчета метрик у нас должны быть ground truth айтемы, про которые мы знаем, что они релевантны пользователю и мы их будем сравнивать с предсказанными. Ground truth айтемы в рекомендательных системах из данного датасета обычно выбираются как реальные положительные взаимодействия пользователя с айтемами в тестовом временном окне.

* Целевая переменная (таргет) — это список или множество айтемов, с которыми пользователь реально взаимодействовал в тестовом (отложенном) временном промежутке.

* Таргет формируется из положительных (значимых) взаимодействий пользователя с айтемами на отложенном периоде, например, кликов, переходов к покупке, лайков и т.п.

* Эти реальные положительные взаимодействия и есть ground truth айтемы, с которыми сравниваются предсказания модели.

* В обучающей выборке у нас есть история взаимодействий пользователя с релевантными товарами за период до теста - на её основе строится модель.

* В тесте модель должна предсказать айтемы, которые user выберет/кликнет/купит. Это и есть задача.

Иными словами, таргет это бинарная метка релевантности айтема пользователю в тестовом периоде, выраженная в виде множества ground truth айтемов. 